# 連合学習チュートリアル（模範解答版・デバッグ用）

**このノートブックは穴埋めをすべて模範解答で埋めた、デバッグ確認用です。**

**連合学習** とは、データを1か所に集めずに、複数の参加者が協力して1つのAIモデルを育てる学習方法です。

このノートブックでは、実際にコードを動かしながらその仕組みを体験します。

やることは大きく3つです。
1. **穴埋め問題**：AIを学習させるコードの空欄を埋めて、自分の手で完成させます。
2. **データの量と分け方の比較**：データをどう持ち寄ると精度が上がるのかを、実験で確かめます。
3. **発展課題**：AIがどの種類を間違えやすいかを調べます（少し難しめ）。

進め方：**上のセルから順番に実行**してください（セルの左の▶を押すか、Shift+Enter）。
まず手を動かすのは「2. 穴埋め」です。

※ このノートブックは計算をすべてCPUで行う設定です（GPUがなくても動きます）。

git test


## 0. 準備：必要なライブラリのインストール
AIの学習に使う道具（ライブラリ）を入れます。**Colabでは毎回必要**で、3〜5分かかります。
このセルを実行して、エラーが出なければ次に進みましょう。


In [ ]:
!pip install -q "flwr[simulation]>=1.18.0" "flwr-datasets[vision]>=0.5.0" torch torchvision matplotlib
# 日本語グラフ用フォント（入らない環境ではフォールバックします）
!pip install -q japanize-matplotlib

In [ ]:
# 日本語フォントの読み込み（japanize-matplotlib が使えない場合はフォント名でフォールバック）
import matplotlib.pyplot as plt
try:
    import japanize_matplotlib  # これだけで日本語が出るようになる
except Exception:
    import matplotlib
    for cand in ["IPAexGothic", "Noto Sans CJK JP", "TakaoPGothic", "MS Gothic"]:
        try:
            matplotlib.rcParams["font.family"] = cand
            break
        except Exception:
            continue
    matplotlib.rcParams["axes.unicode_minus"] = False
print("日本語フォント設定: 完了")

## 1. お手本：1台のパソコンで学習して「データの量と精度」を見る

まずは連合学習を使わず、**1台のパソコンで普通にAIを学習**させてみます。
ここでの目標は、**学習に使うデータ（画像）が多いほど、AIが賢くなる（精度が上がる）** ことを自分の目で確かめることです。

用語のメモ：
- **精度**：AIがどれだけ正解できたかの割合。1.0（＝100%）に近いほど賢い。
- **学習**：たくさんの画像と正解を見せて、AIに見分け方を覚えさせること。

このセルのコードは、あとで挑戦する「2. 穴埋め」の **お手本**にもなっています。
コードの中の「★ TODO① に対応」といった目印が、穴埋めのどの部分に当たるかを示しています。


In [ ]:
"""お手本セル（自己完結・単体学習）: データ数を変えて精度の違いを見る。
task.py より前に置くため、Net もこのセル内に定義する（後の task.py と同じCNN）。
学習ループ・評価には、穴埋め(TODO①②③)に対応する目印コメントを付ける。
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import Compose, Normalize, ToTensor
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner

DEVICE = torch.device("cpu")

# 後の task.py と同じCNN（お手本セル単体で動かすため再掲）
class DemoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1); self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2); self.dropout1 = nn.Dropout(0.25)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1); self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2); self.dropout2 = nn.Dropout(0.25)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64*7*7, 128); self.dropout3 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x)))); x = self.dropout1(x)
        x = self.pool2(F.relu(self.bn2(self.conv2(x)))); x = self.dropout2(x)
        x = self.flatten(x); x = F.relu(self.fc1(x)); x = self.dropout3(x)
        return self.fc2(x)

# 使えるデータセット（名前 → HFのID, 正規化の平均, 標準偏差）
DEMO_DATASETS = {
    "mnist": ("ylecun/mnist", (0.1307,), (0.3081,)),
    "fashion_mnist": ("zalando-datasets/fashion_mnist", (0.2860,), (0.3530,)),
}

# Fashion-MNIST のラベル（0〜9）に対応する日本語の衣類名
FASHION_LABELS_JP = ["Tシャツ/トップ", "ズボン", "プルオーバー", "ドレス", "コート",
                     "サンダル", "シャツ", "スニーカー", "バッグ", "アンクルブーツ"]

def show_samples(dataset_name="mnist", n=10, seed=42):
    """学習に使うデータセットから画像を n 枚ランダムに表示する（ラベル付き）。"""
    import random
    import matplotlib.pyplot as plt
    if dataset_name not in DEMO_DATASETS:
        raise ValueError(f"使えるデータセットは {list(DEMO_DATASETS.keys())} です（指定: {dataset_name}）")
    hf_id, _, _ = DEMO_DATASETS[dataset_name]
    fds = FederatedDataset(dataset=hf_id,
                           partitioners={"train": IidPartitioner(num_partitions=1)})
    part = fds.load_partition(0, "train")
    idxs = random.Random(seed).sample(range(len(part)), n)
    cols = 5
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols * 1.6, rows * 2.0))
    for i, ix in enumerate(idxs):
        ex = part[ix]
        img = ex["image"]
        label = ex["label"]
        if dataset_name == "fashion_mnist":
            label_text = FASHION_LABELS_JP[label]
        else:
            label_text = str(label)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img, cmap="gray")
        plt.axis("off")
        plt.title(label_text, fontsize=9)
    plt.suptitle(f"{dataset_name} のサンプル画像（ランダム {n} 枚）", fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # 全体タイトル用に上を空ける
    plt.subplots_adjust(hspace=0.4)         # 行間を空けてラベルの見切れを防ぐ
    plt.show()

def show_misclassified(net, testloader, device, dataset_name="mnist", n=10, seed=42):
    """誤分類した画像を n 枚ランダムに表示する（正解ラベルと予測ラベル付き）。"""
    import random
    import matplotlib.pyplot as plt
    def label_text(k):
        # Fashion-MNIST は衣類名、MNIST はそのまま数字
        return FASHION_LABELS_JP[k] if dataset_name == "fashion_mnist" else str(k)

    net.to(device); net.eval()
    wrong = []  # (画像, 正解ラベル, 予測ラベル) を集める
    with torch.no_grad():
        for batch in testloader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            predicted = torch.max(net(images).data, 1)[1]
            for img, t, p in zip(images, labels, predicted):
                if t.item() != p.item():
                    wrong.append((img.cpu(), t.item(), p.item()))

    if len(wrong) == 0:
        print("誤分類はありませんでした（全問正解）")
        return

    picks = random.Random(seed).sample(wrong, min(n, len(wrong)))
    cols = 5
    rows = (len(picks) + cols - 1) // cols
    plt.figure(figsize=(cols * 1.7, rows * 2.2))
    for i, (img, t, p) in enumerate(picks):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img.squeeze(), cmap="gray")
        plt.axis("off")
        plt.title(f"正解:{label_text(t)}\n予測:{label_text(p)}", fontsize=8, color="crimson")
    plt.suptitle(f"間違えた画像の例（全{len(wrong)}件中 {len(picks)}枚）", fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.subplots_adjust(hspace=0.5)
    plt.show()

# 画像枚数の上限（実行時間が長くなりすぎないようにする歯止め）
MAX_SAMPLES = 10000
MAX_EPOCHS = 10          # エポック数の上限（実行時間の歯止め）

def _get_data(dataset_name="mnist", seed=42):
    """指定データセットを1まとめで取得し、訓練プールと共通テストセットを作る。"""
    if dataset_name not in DEMO_DATASETS:
        raise ValueError(f"使えるデータセットは {list(DEMO_DATASETS.keys())} です（指定: {dataset_name}）")
    hf_id, mean, std = DEMO_DATASETS[dataset_name]
    fds = FederatedDataset(dataset=hf_id,
                           partitioners={"train": IidPartitioner(num_partitions=1)})
    part = fds.load_partition(0, "train")
    ptt = part.train_test_split(test_size=0.2, seed=seed)
    tfm = Compose([ToTensor(), Normalize(mean, std)])
    def apply(b): b["image"] = [tfm(i) for i in b["image"]]; return b
    ptt = ptt.with_transform(apply)
    return ptt["train"], ptt["test"]

def train_demo(net, loader, test_loader, epochs, lr=0.001, label=""):
    """各エポックの学習後に共通テストセットで評価し、その場で精度を表示する。
    精度の履歴をリストで返す。"""
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    acc_history = []
    for ep in range(epochs):
        net.train()
        for batch in loader:
            images = batch["image"].to(DEVICE); labels = batch["label"].to(DEVICE)
            outputs = net(images); loss = criterion(outputs, labels)
            # ★ TODO① に対応：勾配初期化→逆伝播→パラメータ更新
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        # このエポック終了時点の精度を測り、その場で表示
        acc = test_demo(net, test_loader)
        acc_history.append(acc)
        print(f"    {label}エポック {ep+1}/{epochs} 終了 → 精度 {acc:.4f}")
    return acc_history

def test_demo(net, loader):
    correct, total = 0, 0
    net.eval()
    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE); labels = batch["label"].to(DEVICE)
            outputs = net(images)
            # ★ TODO② に対応：予測ラベルを取り出し、正解数を数える
            predicted = torch.max(outputs.data, 1)[1]
            correct += (predicted == labels).sum().item()
            # --------------------------------------------------
            total += len(labels)
    # ★ TODO③ に対応：精度 = 正解数 / 総数
    accuracy = correct / total if total else 0.0
    return accuracy

def run_demo(sizes=(80, 800, 8000), epochs=3, seed=42, dataset_name="mnist"):
    """枚数ごとに単体学習し、各エポックの精度履歴を返す。
    戻り値: {n: [ep1_acc, ep2_acc, ...]}
    dataset_name: "mnist" または "fashion_mnist"
    """
    torch.manual_seed(seed)
    train_pool, common_test = _get_data(dataset_name, seed)
    # 枚数の上限チェック（プールのサイズと MAX_SAMPLES のどちらか小さい方まで）
    limit = min(MAX_SAMPLES, len(train_pool))
    checked_sizes = []
    for n in sizes:
        if n > limit:
            print(f"※ {n}枚は上限 {limit} を超えるので {limit}枚に制限します")
            n = limit
        checked_sizes.append(n)
    sizes = checked_sizes
    print(f"データセット: {dataset_name}（訓練プール {len(train_pool)}枚・上限 {limit}枚）\n")
    test_loader = DataLoader(common_test, batch_size=64)
    history = {}
    import time
    for n in sizes:
        t = time.time()
        print(f"訓練枚数 {n} で学習中...")
        subset = Subset(train_pool, list(range(n)))
        # 枚数が batch_size 未満でも学習できるよう drop_last を調整
        loader = DataLoader(subset, batch_size=32, shuffle=True, drop_last=(n >= 32))
        net = DemoNet().to(DEVICE)
        accs = train_demo(net, loader, test_loader, epochs)
        history[n] = accs
        print(f"  → 訓練枚数 {n}: 最終精度 {accs[-1]:.4f}  ({time.time()-t:.0f}s)\n")
    return history


### 1-1. 実行：データの枚数を変えて学習する
画像の枚数を **80枚・800枚・8000枚** の3通りに変えて、それぞれAIを学習させます。

**エポック**（同じデータを何回くり返し学習するか）ごとに精度を表示し、折れ線グラフで比べます。
グラフを見ると、「**学習をくり返すほど精度が上がる**」ことと、「**データが多いほど高い精度に届く**」ことの2つが分かります。


In [ ]:
%%time
# まず、学習に使う画像のサンプルを見てみる（MNIST）
show_samples(dataset_name="mnist", n=10)

# 枚数を変えて単体学習し、各エポックの精度を比べる
history = run_demo(sizes=(80, 800, 8000), epochs=3, dataset_name="mnist")

# 折れ線グラフで表示
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
for n, accs in history.items():
    plt.plot(range(1, len(accs)+1), accs, marker="o", label=f"{n}枚")
plt.xlabel("エポック"); plt.ylabel("精度"); plt.ylim(0,1)
plt.title("単体学習：データ数ごとの精度の伸び")
plt.xticks(range(1, max(len(a) for a in history.values())+1))
plt.grid(True); plt.legend(); plt.show()

### 1-2. 自由実習：データセットや枚数を変えて試してみよう

ここからは自分で値を変えて実験してみましょう。次の2つを変えられます。

- **`dataset_name`**：`"mnist"`（手書き数字）か `"fashion_mnist"`（衣類の画像）を選べます。
- **`sizes`**：学習に使う画像の枚数。いくつでも並べられます（例：`(100, 1000, 5000)`）。
  - **枚数の上限は 10000枚** です。超えると自動で10000枚に直されます。

下のセルの数字を書き換えて、何度でも実行してみてください。


In [ ]:
%%time
# ↓ ここを自由に書き換えて実験しよう
dataset_name = "fashion_mnist"   # "mnist" か "fashion_mnist"
sizes = (100, 1000, 5000)        # 画像の枚数（上限 10000）
epochs = 3                       # エポック数（上限 10）

# エポック数の上限チェック
if epochs > MAX_EPOCHS:
    print(f"※ エポック数 {epochs} は上限 {MAX_EPOCHS} を超えるので {MAX_EPOCHS} にします")
    epochs = MAX_EPOCHS

# まず、学習に使う画像のサンプルを見てみる
show_samples(dataset_name=dataset_name, n=10)

# 単体学習を実行
history2 = run_demo(sizes=sizes, epochs=epochs, dataset_name=dataset_name)

# 折れ線グラフで表示
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
for n, accs in history2.items():
    plt.plot(range(1, len(accs)+1), accs, marker="o", label=f"{n}枚")
plt.xlabel("エポック"); plt.ylabel("精度"); plt.ylim(0,1)
plt.title(f"単体学習（{dataset_name}）：データ数ごとの精度の伸び")
plt.xticks(range(1, max(len(a) for a in history2.values())+1))
plt.grid(True); plt.legend(); plt.show()

## 2. 穴埋め：AIを学習・評価するコードを完成させる

AIを動かすために必要なコードの一部が、空欄（`____________________`）になっています。
**3か所のTODO**を埋めて完成させましょう。

お手本（セクション1）のコードを見ながら埋められます。

- **TODO①**：AIに学習させる部分（`train` の中）
- **TODO②③**：AIの成績を測る部分（`test` の中）

空欄を埋めてこのセルを実行し、エラーが出なければ成功です。これで以降の実験がすべて動くようになります。


In [ ]:
# ======================================================================
#  task.py（模範解答版・デバッグ用）
#  3箇所の TODO を埋めると、孤立・集約・連合学習のすべてが動きます。
#  ヒント: 「集約デモ」で動かしたコードと同じ書き方で埋められます。
# ======================================================================
from collections import OrderedDict
import torch
import torch.nn as nn
import torch.nn.functional as F
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Normalize, ToTensor

# データセット登録（正式IDに修正済み・Fashion-MNIST対応）
DATASETS = {
    "mnist": ("ylecun/mnist", (0.1307,), (0.3081,)),
    "fashion_mnist": ("zalando-datasets/fashion_mnist", (0.2860,), (0.3530,)),
}

# CPU固定。GPUを試したい人は force_cpu=False で get_default_device を使う。
def get_default_device(force_cpu=True):
    """計算デバイスを選ぶ。既定はCPU固定。
    force_cpu=False にすると CUDA→Apple Metal(MPS)→CPU の順で自動選択。"""
    if force_cpu:
        return torch.device("cpu")
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32); self.pool1 = nn.MaxPool2d(2, 2); self.dropout1 = nn.Dropout(0.25)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64); self.pool2 = nn.MaxPool2d(2, 2); self.dropout2 = nn.Dropout(0.25)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 7 * 7, 128); self.dropout3 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x)))); x = self.dropout1(x)
        x = self.pool2(F.relu(self.bn2(self.conv2(x)))); x = self.dropout2(x)
        x = self.flatten(x); x = F.relu(self.fc1(x)); x = self.dropout3(x)
        return self.fc2(x)

fds = None
_current_dataset = None

def load_data(partition_id, num_partitions, dataset_name="mnist", random_seed=42, per_client_size=None):
    global fds, _current_dataset
    if dataset_name not in DATASETS:
        raise ValueError(f"未知のデータセット '{dataset_name}'. 使えるのは: {list(DATASETS.keys())}")
    hf_id, mean, std = DATASETS[dataset_name]
    if fds is None or _current_dataset != dataset_name:
        partitioner = IidPartitioner(num_partitions=num_partitions)
        fds = FederatedDataset(dataset=hf_id, partitioners={"train": partitioner})
        _current_dataset = dataset_name

    try:
        partition = fds.load_partition(partition_id, "train")
    except IndexError as e:
        print(f"パーティション {partition_id} を読み込めません。"
              f"partition_id は 0〜{num_partitions - 1} の範囲で指定してください。元のエラー: {e}")
        raise
    except Exception as e:
        print(f"パーティション {partition_id} の読み込み中に予期せぬエラー: {e}")
        raise
    ptt = partition.train_test_split(test_size=0.2, seed=random_seed)
    tfm = Compose([ToTensor(), Normalize(mean, std)])
    def apply(b):
        b["image"] = [tfm(img) for img in b["image"]]; return b
    ptt = ptt.with_transform(apply)
    train_ds = ptt["train"]
    # per_client_size が指定されたら、各クライアントを先頭 per_client_size 枚に絞る
    if per_client_size is not None and per_client_size < len(train_ds):
        train_ds = train_ds.select(range(per_client_size))
    drop_last = len(train_ds) >= 32
    trainloader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=drop_last)
    testloader = DataLoader(ptt["test"], batch_size=32)
    return trainloader, testloader

def train(net, trainloader, epochs, device, learning_rate=0.001):
    net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate)
    criterion = torch.nn.CrossEntropyLoss().to(device)
    net.train()
    losses = []
    for _ in range(epochs):
        running_loss, nb = 0.0, 0
        for batch in trainloader:
            images = batch["image"].to(device); labels = batch["label"].to(device)
            outputs = net(images); loss = criterion(outputs, labels)
            # ==========================================================
            # TODO ①: 学習ループの中核（3行）
            #   勾配を初期化し、誤差を逆伝播し、パラメータを更新してください。
            #   ヒント: optimizer.zero_grad() / loss.backward() / optimizer.step()
            # ----------------------------------------------------------
            optimizer.zero_grad()  # 勾配の初期化
            loss.backward()        # 逆伝播
            optimizer.step()       # パラメータ更新
            # ==========================================================
            running_loss += loss.item(); nb += 1
        losses.append(running_loss / nb if nb else 0.0)
    return sum(losses) / len(losses) if losses else 0.0

def test(net, testloader, device):
    net.to(device)
    criterion = torch.nn.CrossEntropyLoss()
    correct, loss_sum, total = 0, 0.0, 0
    net.eval()
    with torch.no_grad():
        for batch in testloader:
            images = batch["image"].to(device); labels = batch["label"].to(device)
            outputs = net(images)
            loss_sum += criterion(outputs, labels).item() * len(labels); total += len(labels)
            # ==========================================================
            # TODO ②: 予測と正解数のカウント（2行）
            #   出力 outputs から予測ラベルを取り出し、labels と一致した数を
            #   correct に足し込んでください。
            #   ヒント: torch.max(outputs.data, 1)[1] が予測ラベル
            # ----------------------------------------------------------
            predicted = torch.max(outputs.data, 1)[1]  # 予測ラベルを取り出す
            correct += (predicted == labels).sum().item()   # labels と一致した数を加算
            # ==========================================================
    # ==============================================================
    # TODO ③: テスト精度の計算（1行）
    #   正解数 correct と総数 total から精度を求めてください。
    # --------------------------------------------------------------
    accuracy = correct / total if total else 0.0
    # ==============================================================
    return (loss_sum / total if total else 0.0), accuracy

def get_weights(net):
    return [val.cpu().numpy() for _, val in net.state_dict().items()]

def set_weights(net, parameters):
    params_dict = zip(net.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
    net.load_state_dict(state_dict, strict=True)


In [ ]:
# ✅ 答え合わせ（TODO①②③のチェック）
# 上のセルを実行してから、このセルを実行してください。
# 小さなダミーデータで、3つのTODOが正しく書けているか確認します。
import torch as _torch
_device = _torch.device("cpu")

def _check_answers():
    # TODO① 学習ループ：同じデータをくり返し学習して損失が下がるか
    _torch.manual_seed(0)
    _net1 = Net()
    _imgs = _torch.randn(64, 1, 28, 28); _lbls = _torch.randint(0, 10, (64,))
    _loader = [{"image": _imgs, "label": _lbls}]
    _losses = [train(_net1, _loader, epochs=1, device=_device, learning_rate=0.01) for _ in range(15)]
    _ok1 = min(_losses[-5:]) < _losses[0]

    # TODO②③ 予測・精度：test の精度が、手元で計算した正解率と一致するか
    _torch.manual_seed(1)
    _net2 = Net(); _net2.eval()
    _imgs2 = _torch.randn(50, 1, 28, 28); _lbls2 = _torch.randint(0, 10, (50,))
    _loader2 = [{"image": _imgs2, "label": _lbls2}]
    _, _acc_test = test(_net2, _loader2, _device)
    with _torch.no_grad():
        _pred = _net2(_imgs2).argmax(1)
    _acc_manual = (_pred == _lbls2).float().mean().item()
    _ok23 = abs(_acc_test - _acc_manual) < 1e-6
    return _ok1, _ok23

_ok1, _ok23 = _check_answers()
print("TODO① 学習ループ（勾配の初期化・逆伝播・更新）:", "✅ OK" if _ok1 else "❌ NG  もう一度確認しましょう")
print("TODO②③ 予測と精度の計算:", "✅ OK" if _ok23 else "❌ NG  もう一度確認しましょう")
if _ok1 and _ok23:
    print("\nすべて正しく書けています。次に進みましょう。")
else:
    print("\n集約デモのコードと見比べて、TODOの部分を確認してみてください。")

## 3. 連合学習を動かす準備
連合学習の本体です。このセルは、連合学習を実行するための関数をまとめて用意します。

（セクション2の穴埋めが正しく埋まっていれば、そのまま実行できます。）


In [ ]:
import torch
from flwr.client import ClientApp, NumPyClient
from flwr.server import ServerApp, ServerAppComponents, ServerConfig
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation
from flwr.common import Context, ndarrays_to_parameters
# （task の関数は上のセルで定義済み）

# history collector (in-memory, no Excel)
history = {"round": [], "train_loss": [], "eval_loss": [], "accuracy": []}
client_sizes = {}  # partition_id -> 訓練サンプル数

def run_federated(dataset_name="mnist", num_clients=5, num_rounds=3,
                  local_epochs=1, learning_rate=0.001, fraction_fit=1.0, seed=42, per_client_size=None):
    for k in history: history[k].clear()
    DEVICE = torch.device("cpu")

    class FlowerClient(NumPyClient):
        def __init__(self, net, trainloader, valloader, local_epochs):
            self.net=net; self.trainloader=trainloader; self.valloader=valloader
            self.local_epochs=local_epochs; self.device=DEVICE; self.net.to(DEVICE)
        def fit(self, parameters, config):
            set_weights(self.net, parameters)
            lr=float(config.get("learning_rate", learning_rate))
            avg=train(self.net, self.trainloader, self.local_epochs, self.device, learning_rate=lr)
            return get_weights(self.net), len(self.trainloader.dataset), {"train_loss":avg}
        def evaluate(self, parameters, config):
            set_weights(self.net, parameters)
            loss, acc = test(self.net, self.valloader, self.device)
            return loss, len(self.valloader.dataset), {"accuracy":acc, "loss":loss}

    def client_fn(context: Context):
        pid=context.node_config["partition-id"]; npart=context.node_config["num-partitions"]
        tl, vl = load_data(pid, npart, dataset_name=dataset_name, random_seed=seed, per_client_size=per_client_size)
        return FlowerClient(Net(), tl, vl, local_epochs).to_client()

    client_app = ClientApp(client_fn=client_fn)

    def weighted_average(metrics):
        tot=sum(n for n,_ in metrics)
        if tot==0: return {"accuracy":0.0,"loss":0.0}
        acc=sum(n*m.get("accuracy",0) for n,m in metrics)/tot
        loss=sum(n*m.get("loss",0) for n,m in metrics)/tot
        return {"accuracy":acc,"loss":loss}
    def fit_agg(metrics):
        tot=sum(n for n,_ in metrics)
        return {"train_loss": sum(n*m.get("train_loss",0) for n,m in metrics)/tot if tot else 0.0}

    class TrackingFedAvg(FedAvg):
        def aggregate_fit(self, rnd, results, failures):
            p,m=super().aggregate_fit(rnd,results,failures)
            self._last_train_loss=m.get("train_loss") if m else None
            return p,m
        def aggregate_evaluate(self, rnd, results, failures):
            loss,m=super().aggregate_evaluate(rnd,results,failures)
            history["round"].append(rnd)
            history["train_loss"].append(getattr(self,"_last_train_loss",None))
            history["eval_loss"].append(m.get("loss") if m else None)
            history["accuracy"].append(m.get("accuracy") if m else None)
            print(f"  [round {rnd}] train_loss={getattr(self,'_last_train_loss',None):.4f}  eval_loss={m.get('loss'):.4f}  accuracy={m.get('accuracy'):.4f}")
            return loss,m

    def server_fn(context: Context):
        params=ndarrays_to_parameters(get_weights(Net()))
        strat=TrackingFedAvg(
            fraction_fit=fraction_fit, fraction_evaluate=1.0,
            min_available_clients=num_clients,
            min_fit_clients=max(1,int(num_clients*fraction_fit)),
            min_evaluate_clients=num_clients,
            initial_parameters=params,
            evaluate_metrics_aggregation_fn=weighted_average,
            fit_metrics_aggregation_fn=fit_agg,
            on_fit_config_fn=lambda r: {"learning_rate":learning_rate},
        )
        return ServerAppComponents(strategy=strat, config=ServerConfig(num_rounds=num_rounds))

    server_app = ServerApp(server_fn=server_fn)
    run_simulation(
        server_app=server_app, client_app=client_app,
        num_supernodes=num_clients,
        backend_config={"client_resources":{"num_cpus":1,"num_gpus":0.0}},
    )
    result = dict(history)
    result["client_sizes"] = dict(sorted(client_sizes.items()))
    return result

def get_client_sizes(dataset_name="mnist", num_clients=5, seed=42):
    """各クライアントの訓練サンプル数を集計して返す（メインプロセスで計算）。"""
    sizes = {}
    for pid in range(num_clients):
        tl, _ = load_data(pid, num_clients, dataset_name=dataset_name, random_seed=seed)
        sizes[pid] = len(tl.dataset)
    return sizes



In [ ]:
"""クライアント別サンプル数を表で表示するヘルパー"""

def show_client_sizes(client_sizes, title="クライアント別サンプル数"):
    """client_sizes: {partition_id: num_samples}
    各クライアントの訓練サンプル数を表（テキスト）で表示する。
    """
    ids = list(client_sizes.keys())
    sizes = [client_sizes[i] for i in ids]
    print(f"=== {title} ===")
    print(f"{'クライアント':>10s} {'サンプル数':>10s}")
    for i, n in zip(ids, sizes):
        print(f"{i:>10d} {n:>10d}")
    print(f"{'合計':>10s} {sum(sizes):>10d}")


## 4. 連合学習を実行してみる
複数の参加者（**クライアント**＝データを持つ各パソコンのこと）が協力して、1つのAIを育てます。

**ラウンド**（参加者が学習結果を持ち寄って、AIを1回まとめ直す単位）をくり返すごとに、
AIの精度が上がっていけば成功です。


In [ ]:
%%time
h = run_federated(dataset_name="mnist", num_clients=5, num_rounds=3, per_client_size=1000)
print("最終精度:", h["accuracy"][-1])

### 4-1. 精度の変化をグラフで見る
ラウンドをくり返すたびに精度がどう変わったかを、折れ線グラフで確認します。


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.plot(h["round"], h["accuracy"], marker="o")
plt.xlabel("ラウンド"); plt.ylabel("精度"); plt.ylim(0,1)
plt.title("連合学習：ラウンドごとの精度"); plt.grid(True); plt.show()

### 4-2. 各参加者が持っているデータの数（表）
今回は全参加者にデータを均等に配っているので、各クライアントの枚数はほぼ同じになります。
（次のセクションでは、わざと配り方を変えて違いを見ます）


In [ ]:
sizes = get_client_sizes(dataset_name="mnist", num_clients=5)
show_client_sizes(sizes, title="通常の連合学習：クライアント別サンプル数")

---
## 5. 比べてみる：データの「量」と「分け方」で精度はどう変わる？

- **設定①（分けるだけ）**：800枚の画像を、みんなで分け合う（1人あたりは少なくなる）
- **設定②（みんなが同じ量を持つ）**：参加者全員がそれぞれ異なる800枚の画像を持つ（全体の枚数は増える）


In [ ]:
"""設定①②: 連合学習でのデータ量比較（解釈A）
基準 = 孤立1クライアント分の枚数 base_size
 設定①(総量固定で分割): 総量 base_size を N分割 → 1人 base_size/N
 設定②(1人あたり固定)  : 各クライアントが base_size 枚 → 総量 base_size*N
"""
import torch
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import Compose, Normalize, ToTensor
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner
from flwr.client import ClientApp, NumPyClient
from flwr.server import ServerApp, ServerAppComponents, ServerConfig
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation
from flwr.common import Context, ndarrays_to_parameters
# （上のセルで定義済み）

def _load_pool(dataset_name, seed):
    """1パーティション(=全データ)をまとめて取得し、train/common-testに分ける"""
    hf_id, mean, std = DATASETS[dataset_name]
    fds = FederatedDataset(dataset=hf_id, partitioners={"train": IidPartitioner(num_partitions=1)})
    part = fds.load_partition(0, "train")
    ptt = part.train_test_split(test_size=0.2, seed=seed)
    tfm = Compose([ToTensor(), Normalize(mean, std)])
    def apply(b):
        b["image"] = [tfm(img) for img in b["image"]]; return b
    ptt = ptt.with_transform(apply)
    return ptt["train"], ptt["test"]

def run_fl_datasplit(dataset_name="mnist", num_clients=5, base_size=2000,
                     mode="split", num_rounds=3, local_epochs=1,
                     learning_rate=0.001, seed=42):
    device = torch.device("cpu")
    torch.manual_seed(seed)
    train_pool, common_test = _load_pool(dataset_name, seed)

    # 各クライアントが持つインデックスを決める
    if mode == "split":          # 設定①: 総量 base_size を分割
        per = base_size // num_clients
        idx_all = list(range(base_size))
        client_idx = [idx_all[i*per:(i+1)*per] for i in range(num_clients)]
    elif mode == "each":         # 設定②: 各自 base_size 枚（重ならないように連続割当）
        client_idx = [list(range(i*base_size, (i+1)*base_size)) for i in range(num_clients)]
    else:
        raise ValueError(mode)

    history = {"round": [], "accuracy": []}
    test_loader = DataLoader(common_test, batch_size=64)

    class FlowerClient(NumPyClient):
        def __init__(self, subset):
            self.net = Net()
            self.loader = DataLoader(subset, batch_size=32, shuffle=True,
                                     drop_last=len(subset) >= 32)
        def fit(self, parameters, config):
            set_weights(self.net, parameters)
            train(self.net, self.loader, local_epochs, device, learning_rate=learning_rate)
            return get_weights(self.net), len(self.loader.dataset), {}
        def evaluate(self, parameters, config):
            set_weights(self.net, parameters)
            loss, acc = test(self.net, DataLoader(common_test, batch_size=64), device)
            return loss, len(common_test), {"accuracy": acc}

    def client_fn(ctx: Context):
        pid = ctx.node_config["partition-id"]
        subset = Subset(train_pool, client_idx[pid])
        return FlowerClient(subset).to_client()

    def agg(metrics):
        tot = sum(n for n, _ in metrics)
        return {"accuracy": sum(n*m.get("accuracy",0) for n,m in metrics)/tot if tot else 0.0}

    class T(FedAvg):
        def aggregate_evaluate(self, rnd, results, failures):
            loss, m = super().aggregate_evaluate(rnd, results, failures)
            history["round"].append(rnd); history["accuracy"].append(m.get("accuracy"))
            print(f"  [{mode}] round {rnd}: accuracy={m.get('accuracy'):.4f}")
            return loss, m

    def server_fn(ctx: Context):
        strat = T(fraction_fit=1.0, fraction_evaluate=1.0,
                  min_available_clients=num_clients, min_fit_clients=num_clients,
                  min_evaluate_clients=num_clients,
                  initial_parameters=ndarrays_to_parameters(get_weights(Net())),
                  evaluate_metrics_aggregation_fn=agg)
        return ServerAppComponents(strategy=strat, config=ServerConfig(num_rounds=num_rounds))

    run_simulation(server_app=ServerApp(server_fn=server_fn),
                   client_app=ClientApp(client_fn=client_fn),
                   num_supernodes=num_clients,
                   backend_config={"client_resources": {"num_cpus": 1, "num_gpus": 0.0}})
    total = sum(len(c) for c in client_idx)
    client_sizes = {i: len(c) for i, c in enumerate(client_idx)}
    return {"mode": mode, "per_client": len(client_idx[0]), "total": total,
            "client_sizes": client_sizes,
            "final_accuracy": history["accuracy"][-1] if history["accuracy"] else None}


### 5-1. 設定①（分けるだけ）を実行
1人分のデータをみんなで分け合います。各参加者の持つ枚数は少なくなります。


In [ ]:
%%time
r_split = run_fl_datasplit(dataset_name="mnist", num_clients=5, base_size=800, mode="split", num_rounds=3)
print(r_split["mode"], "final_accuracy=", r_split["final_accuracy"])
show_client_sizes(r_split["client_sizes"], title="設定①(split)：クライアント別サンプル数")

### 5-2. 設定②（みんなが同じ量を持つ）を実行
参加者全員が1人分と同じ枚数を持ちます。全体のデータ量はそのぶん増えます。


In [ ]:
%%time
r_each = run_fl_datasplit(dataset_name="mnist", num_clients=5, base_size=800, mode="each", num_rounds=3)
print(r_each["mode"], "final_accuracy=", r_each["final_accuracy"])
show_client_sizes(r_each["client_sizes"], title="設定②(each)：クライアント別サンプル数")

### 5-3. 結果を並べて比べる
設定①（分けるだけ）は総枚数が変わらないので精度はあまり伸びず、
設定②（みんなが同じ量）は全体の枚数が増えるので精度が伸びる、という違いが見えるはずです。
連合学習は「参加者がそれぞれ十分なデータを持ち寄るほど効果が大きい」と分かります。


In [ ]:
print(f"{'設定':18s} {'1人あたり':>8s} {'総量':>7s} {'精度':>8s}")
print(f"{'設定①(分割)':18s} {r_split['per_client']:8d} {r_split['total']:7d} {r_split['final_accuracy']:8.4f}")
print(f"{'設定②(各自固定)':18s} {r_each['per_client']:8d} {r_each['total']:7d} {r_each['final_accuracy']:8.4f}")

### 5-4. 自由実習：連合学習のいろいろな条件を試してみよう

ここからは連合学習の条件を自分で変えて実験してみましょう。次の4つを変えられます。

- **`dataset_name`**：`"mnist"`（手書き数字）か `"fashion_mnist"`（衣類の画像）
- **`per_client_size`**：参加者1人あたりの画像の枚数（上限 2000）
- **`num_clients`**：参加者（クライアント）の人数（上限 10）
- **`num_rounds`**：学習結果を持ち寄る回数（ラウンド数・上限 5）

学習に使う画像の総数は「**1人あたりの枚数 × 人数**」で決まります。
下のセルの値を書き換えて、何度でも実行してみてください。
たとえば「人数を増やすと精度はどうなる？」「1人あたりの枚数を増やすと？」などを確かめられます。

**注意**：連合学習は計算が重く、1回の実行に数分かかります。
人数や枚数を大きくすると時間が伸びるので、まずは小さめの値から試すのがおすすめです。

In [ ]:
%%time
# ↓ ここを自由に書き換えて実験しよう
dataset_name    = "fashion_mnist"   # "mnist" か "fashion_mnist"
per_client_size = 1000              # 参加者1人あたりの枚数（上限 2000）
num_clients     = 5                 # 参加者の人数（上限 10）
num_rounds      = 3                 # ラウンド数（上限 5）

# 時間がかかりすぎないための上限チェック
MAX_PER_CLIENT, MAX_CLIENTS, MAX_ROUNDS = 2000, 10, 5
if per_client_size > MAX_PER_CLIENT:
    print(f"※ 1人あたりの枚数 {per_client_size} は上限 {MAX_PER_CLIENT} を超えるので {MAX_PER_CLIENT} にします")
    per_client_size = MAX_PER_CLIENT
if num_clients > MAX_CLIENTS:
    print(f"※ 人数 {num_clients} は上限 {MAX_CLIENTS} を超えるので {MAX_CLIENTS} にします")
    num_clients = MAX_CLIENTS
if num_rounds > MAX_ROUNDS:
    print(f"※ ラウンド数 {num_rounds} は上限 {MAX_ROUNDS} を超えるので {MAX_ROUNDS} にします")
    num_rounds = MAX_ROUNDS

# まず、学習に使う画像のサンプルを見てみる
show_samples(dataset_name=dataset_name, n=10)

# 連合学習を実行（各参加者が per_client_size 枚ずつ持つ）
r_free = run_fl_datasplit(dataset_name=dataset_name, num_clients=num_clients,
                          base_size=per_client_size, mode="each", num_rounds=num_rounds)
total = per_client_size * num_clients
print(f"\n設定: 1人あたり {per_client_size}枚 × {num_clients}人 = 総量 {total}枚 / {num_rounds}ラウンド")
print(f"最終精度: {r_free['final_accuracy']:.4f}")

# 各クライアントが持っているデータの数を表で表示
show_client_sizes(r_free["client_sizes"],
                  title="自由実習：クライアント別サンプル数")

---
## 6. 発展課題（穴埋め）：どのクラスを間違えやすい？

全体の精度だけでは「どの種類（クラス）でつまずいているか」は分かりません。
そこで、**クラスごとの正答率**と、**混同行列**（どのクラスをどのクラスと取り違えたかを表にしたもの）を作ります。

下のコードには **5か所のTODO** があります。各TODOには「何をするか」だけ書いてあるので、コードは自分で考えて埋めてください。

**ヒント**
- 出発点は、セクション2の穴埋め `task.py` の **TODO②**（予測を取り出して正解数を数える部分）です。発展課題はこれを「クラスごと」に数えるよう拡張します。
- **TODO③④**（クラスごとの正解数・総数を数える）は、`task.py` のTODO②を1クラスずつに分けたものと考えると分かりやすいです。
- **TODO①⑤**（混同行列）は、`confusion[正解のクラス][予測したクラス]` のように2次元の表のマス目を1つずつ増やしていきます。
- 調べるときのキーワード：「**Python リスト内包表記 2次元**」「**混同行列 confusion matrix とは**」「**PyTorch torch.max 予測ラベル**」


In [ ]:
# ======================================================================
#  アドバンスド課題（模範解答版・デバッグ用）: クラスごとの正答率 ＋ 混同行列
#  5箇所の TODO を埋めてください。かなり難しめです。
#  各 TODO には「何をするか」だけ書いてあります。コードは自分で考えること。
#  ヒント: 穴埋め task.py の TODO② が出発点。それを「クラス別」に拡張する。
# ======================================================================
import torch

def per_class_accuracy(net, testloader, device, num_classes=10):
    """クラスごとの正答率と混同行列を計算して返す。
    戻り値: (per_class_acc, confusion, overall_acc)
      - per_class_acc[k] : クラス k の正答率（float）
      - confusion[t][p]  : 正解が t で予測が p だった件数（int）
      - overall_acc      : 全体の正答率（float）
    """
    net.to(device); net.eval()
    correct = [0] * num_classes
    total = [0] * num_classes

    # ================================================================
    # TODO ①: 混同行列を初期化する
    #   num_classes 行 × num_classes 列 の、すべて 0 の二次元リストを作る。
    #   （confusion[t][p] に「正解t・予測p」の件数を入れていく）
    # ----------------------------------------------------------------
    confusion = [[0] * num_classes for _ in range(num_classes)]
    # ================================================================

    with torch.no_grad():
        for batch in testloader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            outputs = net(images)

            # ============================================================
            # TODO ②: 予測ラベルを取り出す
            #   モデル出力 outputs から、各サンプルの予測クラスを取り出す。
            # ------------------------------------------------------------
            predicted = torch.max(outputs.data, 1)[1]
            # ============================================================

            for label, pred in zip(labels, predicted):
                t = label.item()
                p = pred.item()

                # ====================================================
                # TODO ③: 正解クラス t の総数を 1 増やす
                # ----------------------------------------------------
                total[t] += 1
                # ====================================================

                # ====================================================
                # TODO ④（2行）: 予測が正解と一致していたら、
                #         正解クラス t の正解数を 1 増やす（if文 + カウントの2行）
                # ----------------------------------------------------
                if t == p:
                    correct[t] += 1
                # ====================================================

                # ====================================================
                # TODO ⑤: 混同行列の該当要素（正解t・予測p）を 1 増やす
                # ----------------------------------------------------
                confusion[t][p] += 1
                # ====================================================

    per_class_acc = [correct[k] / total[k] if total[k] else 0.0 for k in range(num_classes)]
    overall_acc = sum(correct) / sum(total) if sum(total) else 0.0
    return per_class_acc, confusion, overall_acc

# ✅ 答え合わせ（TODO①〜⑤のチェック）
# 小さなダミーデータで per_class_accuracy を動かし、
# 手元で計算した正しい混同行列・正答率と一致するか確認します。
def _check_per_class():
    torch.manual_seed(2)
    _net = Net(); _net.eval()
    _imgs = torch.randn(40, 1, 28, 28); _lbls = torch.randint(0, 10, (40,))
    _loader = [{"image": _imgs, "label": _lbls}]
    _pca, _conf, _overall = per_class_accuracy(_net, _loader, torch.device("cpu"), num_classes=10)

    # 手元で正しい答えを計算
    with torch.no_grad():
        _pred = _net(_imgs).argmax(1)
    _conf_ref = [[0]*10 for _ in range(10)]
    _correct = [0]*10; _total = [0]*10
    for _t, _p in zip(_lbls.tolist(), _pred.tolist()):
        _conf_ref[_t][_p] += 1; _total[_t] += 1
        if _t == _p: _correct[_t] += 1
    _pca_ref = [_correct[k]/_total[k] if _total[k] else 0.0 for k in range(10)]
    _overall_ref = sum(_correct) / sum(_total)

    _ok_conf = (_conf == _conf_ref)
    _ok_pca = all(abs(a-b) < 1e-9 for a, b in zip(_pca, _pca_ref))
    _ok_overall = abs(_overall - _overall_ref) < 1e-9
    return _ok_conf, _ok_pca, _ok_overall

_oc, _op, _oo = _check_per_class()
print("TODO①⑤ 混同行列:", "✅ OK" if _oc else "❌ NG")
print("TODO②③④ クラス別正答率:", "✅ OK" if (_op and _oo) else "❌ NG")
if _oc and _op and _oo:
    print("\nすべて正しく書けています。下のセルで結果を見てみましょう。")
else:
    print("\nTODOの番号をもう一度確認してみてください。")


### 6-1. 結果を見る：クラス別の正答率と混同行列
埋めた関数を使って、クラスごとの正答率（棒グラフ）と混同行列（色の濃さで件数を表した図）を表示します。
混同行列の見方：縦が「本当の正解」、横が「AIの予測」。**対角線が濃いほど正しく当てられている**ことを表します。


In [ ]:
import torch
from torch.utils.data import DataLoader
tl, vl = load_data(0, 5, "mnist")
net = Net(); train(net, tl, 2, get_default_device())
per_class_acc, confusion, overall = per_class_accuracy(net, vl, get_default_device())
print("全体の正答率:", round(overall, 4))

import matplotlib.pyplot as plt
# クラス別正答率（棒グラフ）
plt.figure(figsize=(7,4)); plt.bar(range(10), per_class_acc)
plt.xticks(range(10)); plt.ylim(0,1); plt.xlabel("クラス"); plt.ylabel("正答率")
plt.title("クラスごとの正答率")
for i,a in enumerate(per_class_acc): plt.text(i, a+0.02, f"{a:.2f}", ha="center", fontsize=8)
plt.show()

# 混同行列（ヒートマップ）
import numpy as np
conf = np.array(confusion)
plt.figure(figsize=(6,5))
plt.imshow(conf, cmap="Blues")
plt.colorbar(label="件数")
plt.xlabel("予測クラス"); plt.ylabel("正解クラス")
plt.title("混同行列")
plt.xticks(range(10)); plt.yticks(range(10))
for t in range(10):
    for p in range(10):
        if conf[t][p] > 0:
            plt.text(p, t, conf[t][p], ha="center", va="center", fontsize=7,
                     color="white" if conf[t][p] > conf.max()/2 else "black")
plt.tight_layout(); plt.show()

# 実際に間違えた画像を10枚見てみる（正解と予測のラベル付き）
show_misclassified(net, vl, get_default_device(), dataset_name="mnist", n=10)

## 7. 考察課題

ここまでの実習をふりかえって、次の問いを考えてみましょう。
（正解が1つに決まる問題ではありません。自分の言葉で説明してみてください。）

**問1.** 1台だけで学習したとき（お手本）、使う画像の枚数を増やすと精度はどう変わりましたか。
その理由を、AIが学習するしくみと結びつけて説明してみましょう。

**問2.** 比べてみる（第5章）で、設定①（みんなで分け合う）と設定②（みんなが同じ量を持つ）では、
精度に違いが出ました。なぜこのような違いが生まれたのか、自分の言葉で説明してみましょう。

**問3.** 発展課題（第6章）で、クラス（数字や衣類の種類）によって正答率に差がありました。
間違えやすかったクラスはどれで、なぜ間違えやすいのか、混同行列を見ながら考えてみましょう。

**問4.** 水平連合学習が役に立ちそうな身近な例を、このチュートリアル以外で1つ考えてみましょう。
複数の人や組織が「同じ種類のデータ」を別々に持っている状況です。
誰がどんなデータを持ち寄り、何を予測・分類すると便利か、説明してみましょう。

---

### 解答例・着眼点（模範解答版）

**問1の着眼点.**
枚数を増やすほど精度は上がる傾向（ただし、ある程度で頭打ちになる）。
AIはたくさんの例を見るほど、見たことのないデータにも対応できる一般的な特徴をつかめるようになるため。
データが少ないと、たまたまの偏りを覚えてしまい（過学習）、テストでうまくいかないことにも触れられるとよい。

**問2の着眼点.**
設定①は全体の枚数が変わらないまま分けるだけなので、1人あたりが減り、精度はあまり伸びない。
設定②は参加者が増えるほど全体の枚数が増えるので、精度が伸びる。
「連合学習は、参加者がそれぞれ十分なデータを持ち寄るほど効果が大きい」とまとめられればよい。

**問3の着眼点.**
データによるが、手書き数字なら 4 と 9、3 と 5 など形の似た数字、
衣類なら シャツ・コート・プルオーバーなど見た目の似たカテゴリが間違えやすい。
混同行列で「正解はAなのにBと予測した件数」が多いマスに着目できていればよい。
「形や見た目が似ているものほど区別が難しい」と説明できると十分。

**問4の着眼点.**
正解は一つではない。複数の人・組織が同じ種類のデータを別々に持つ例であればよい。
例：複数の病院が、それぞれの患者の同じ検査画像を持ち、共同で病気を判定するモデルを作る。
例：複数のスマホが、それぞれの利用者の文字入力履歴を持ち、共同で予測変換を賢くする。
例：複数の店舗が、それぞれの来客データを持ち、共同で売れ筋を予測する。
「データを1か所に集めずに学習したい理由（プライバシー・通信量）」にも触れられていれば、なお良い。
